In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from river import drift, linear_model, metrics, optim

In [ ]:
df = pd.read_csv('../data/LSudden_3.csv', header=None).iloc[:, 0]
X_col = df.columns[:5]
y_col = df.columns[5]
X = df[X_col]
y = df[y_col]

In [ ]:
df.describe()

In [ ]:
def learn_many(model, df):
    for _, row in df.iterrows():
        X_i = dict(row[X_col])
        y_i = row[y_col]
        model.learn_one(X_i, y_i)

In [ ]:
train_size = 1000
model = linear_model.SoftmaxRegression()
learn_many(model, df[:train_size])

In [ ]:
adwin = drift.ADWIN()
accuracy = metrics.Accuracy()
accuracies = []
opt = optim.losses.CrossEntropy()

for i, row in df[train_size:].iterrows():
    X_i = row[X_col]
    y_i = row[y_col]

    y_pred_proba = model.predict_proba_one(X_i)
    loss = opt(y_i, y_pred_proba)
    adwin.update(loss)

    y_pred = model.predict_one(X_i)
    accuracy.update(y_i, y_pred)
    accuracies.append(accuracy.get())

    if adwin.drift_detected:
        print(f'Drift detected @ sample #{train_size + i + 1}')
        print('Retraining model')
        learn_many(model, df[i + 1: train_size + i + 1])

The output of the first experiment I made:

```
Drift detected @ sample #201168
Retraining model
Drift detected @ sample #351952
Retraining model
Drift detected @ sample #401104
Retraining model
Drift detected @ sample #601520
Retraining model
Drift detected @ sample #801072
Retraining model
```

The documentation of the dataset stated that an abrupt drift occurs every 200k points.
This suggests that even this very simple model did quite well thanks to ADWIN.

In [ ]:
plt.plot(accuracies)
plt.title('Accuracy')
plt.show()